<a href="https://colab.research.google.com/github/pluplu68/agent-lab/blob/main/HF-LLM-course/create_own_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating your own dataset

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!apt install git-lfs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 3,908 kB of archives.
After this operation, 11.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 git-lfs amd64 3.4.1-1ubuntu0.4 [3,908 kB]
Fetched 3,908 kB in 1s (3,672 kB/s)
Selecting previously unselected package git-lfs.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../git-lfs_3.4.1-1ubuntu0.4_amd64.deb ...
Unpacking git-lfs (3.4.1-1ubuntu0.4) ...
Setting up git-lfs (3.4.1-1ubuntu0.4) ...
Processing triggers for man-db (2.12.0-4build2) ...


You will need to setup git, adapt your email and name in the following cell.

In [2]:
!git config --global user.email "tianyili2020@gmail.com"
!git config --global user.name "Tiana"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
!pip install requests

In [5]:
import requests

url = "https://api.github.com/repos/huggingface/datasets/issues?page=1&per_page=1"
response = requests.get(url)

In [6]:
response.status_code

200

In [7]:
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/9080',
  'repository_url': 'https://api.github.com/repos/huggingface/datasets',
  'labels_url': 'https://api.github.com/repos/huggingface/datasets/issues/9080/labels{/name}',
  'comments_url': 'https://api.github.com/repos/huggingface/datasets/issues/9080/comments',
  'events_url': 'https://api.github.com/repos/huggingface/datasets/issues/9080/events',
  'html_url': 'https://github.com/huggingface/datasets/pull/9080',
  'id': 6476408815,
  'node_id': 'PR_kwDODunzps8AAAABQOa1RA',
  'number': 9080,
  'title': 'Reuse unchanged columns in Dataset.map when input_columns is specified',
  'user': {'login': 'GuilhermeLopesBertacini',
   'id': 167634752,
   'node_id': 'U_kgDOCf3nQA',
   'avatar_url': 'https://avatars.githubusercontent.com/u/167634752?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/GuilhermeLopesBertacini',
   'html_url': 'https://github.com/GuilhermeLopesBertacini',
   'followers_url': 'https

In [11]:
# GITHUB_TOKEN = github_pat_XXXX  # Copy your GitHub token here
# headers = {"Authorization": f"token {GITHUB_TOKEN}"}

from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}"
}

In [18]:
import time
import math
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm


def fetch_issues(
    owner="huggingface",
    repo="datasets",
    num_issues=10_000,
    rate_limit=5_000,
    issues_path=Path("."),
):
    if not issues_path.is_dir():
        issues_path.mkdir(exist_ok=True)

    batch = []
    all_issues = []
    per_page = 100  # Number of issues to return per page
    num_pages = math.ceil(num_issues / per_page)
    base_url = "https://api.github.com/repos"

    # for page in tqdm(range(num_pages)):
    for page in tqdm(range(1, num_pages + 1)):
        query = f"issues?page={page}&per_page={per_page}&state=all"

        response = requests.get(
            f"{base_url}/{owner}/{repo}/{query}",
            headers=headers
        )
        response.raise_for_status()

        batch.extend(response.json())
        # # Query with state=all to get both open and closed issues
        # query = f"issues?page={page}&per_page={per_page}&state=all"
        # issues = requests.get(f"{base_url}/{owner}/{repo}/{query}", headers=headers)
        # batch.extend(issues.json())

        # if len(batch) > rate_limit and len(all_issues) < num_issues:
        #     all_issues.extend(batch)
        #     batch = []  # Flush batch for next time period
        #     print(f"Reached GitHub rate limit. Sleeping for one hour ...")
        #     time.sleep(60 * 60 + 1)

    all_issues.extend(batch)
    df = pd.DataFrame.from_records(all_issues)
    df.to_json(f"{issues_path}/{repo}-issues.jsonl", orient="records", lines=True)
    print(
        f"Downloaded all the issues for {repo}! Dataset stored at {issues_path}/{repo}-issues.jsonl"
    )

In [19]:
import requests

r = requests.get(
    "https://api.github.com/rate_limit",
    headers=headers
)

print(r.status_code)
print(r.json().get("resources", {}).get("core"))

200
{'limit': 5000, 'used': 53, 'remaining': 4947, 'reset': 1795065883}


In [21]:
# Depending on your internet connection, this can take several minutes to run...
# fetch_issues()
fetch_issues(num_issues=9_900)

  0%|          | 0/99 [00:00<?, ?it/s]

Downloaded all the issues for datasets! Dataset stored at ./datasets-issues.jsonl


In [25]:
from datasets import load_dataset
from pathlib import Path

file_path = "datasets-issues.jsonl"

issues_dataset = load_dataset("json", data_files="datasets-issues.jsonl", split="train", chunksize=Path(file_path).stat().st_size + 1024)
issues_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment'],
    num_rows: 8838
})

In [26]:
sample = issues_dataset.shuffle(seed=666).select(range(3))

# Print out the URL and pull request entries
for url, pr in zip(sample["html_url"], sample["pull_request"]):
    print(f">> URL: {url}")
    print(f">> Pull request: {pr}\n")

>> URL: https://github.com/huggingface/datasets/pull/2392
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/2392', 'html_url': 'https://github.com/huggingface/datasets/pull/2392', 'diff_url': 'https://github.com/huggingface/datasets/pull/2392.diff', 'patch_url': 'https://github.com/huggingface/datasets/pull/2392.patch', 'merged_at': datetime.datetime(2021, 5, 28, 11, 37, 32)}

>> URL: https://github.com/huggingface/datasets/pull/3130
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/3130', 'html_url': 'https://github.com/huggingface/datasets/pull/3130', 'diff_url': 'https://github.com/huggingface/datasets/pull/3130.diff', 'patch_url': 'https://github.com/huggingface/datasets/pull/3130.patch', 'merged_at': None}

>> URL: https://github.com/huggingface/datasets/pull/3180
>> Pull request: {'url': 'https://api.github.com/repos/huggingface/datasets/pulls/3180', 'html_url': 'https://github.com/huggingface/datasets/pull/3180', 'd

In [27]:
issues_dataset = issues_dataset.map(
    lambda x: {"is_pull_request": False if x["pull_request"] is None else True}
)

Map:   0%|          | 0/8838 [00:00<?, ? examples/s]

In [ ]:
# Try it out! Calculate the average time it takes to close issues in 🤗 Datasets.

# Keep only closed issues, excluding PRs
closed_issues = issues_dataset.filter(
    lambda x: (
        not x["is_pull_request"]
        and x["closed_at"] is not None
    )
)

# Convert to Pandas
df = closed_issues.to_pandas()

# Compute closing time
df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df["closed_at"] = pd.to_datetime(df["closed_at"], utc=True)

df["days_to_close"] = (
    df["closed_at"] - df["created_at"]
).dt.total_seconds() / 86400

print("Average days to close:", df["days_to_close"].mean())

In [28]:
issue_number = 2792
url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
response = requests.get(url, headers=headers)
response.json()

[{'url': 'https://api.github.com/repos/huggingface/datasets/issues/comments/897594128',
  'html_url': 'https://github.com/huggingface/datasets/pull/2792#issuecomment-897594128',
  'issue_url': 'https://api.github.com/repos/huggingface/datasets/issues/2792',
  'id': 897594128,
  'node_id': 'IC_kwDODunzps41gDMQ',
  'user': {'login': 'bhavitvyamalik',
   'id': 19718818,
   'node_id': 'MDQ6VXNlcjE5NzE4ODE4',
   'avatar_url': 'https://avatars.githubusercontent.com/u/19718818?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/bhavitvyamalik',
   'html_url': 'https://github.com/bhavitvyamalik',
   'followers_url': 'https://api.github.com/users/bhavitvyamalik/followers',
   'following_url': 'https://api.github.com/users/bhavitvyamalik/following{/other_user}',
   'gists_url': 'https://api.github.com/users/bhavitvyamalik/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/bhavitvyamalik/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/

In [29]:
def get_comments(issue_number):
    url = f"https://api.github.com/repos/huggingface/datasets/issues/{issue_number}/comments"
    response = requests.get(url, headers=headers)
    return [r["body"] for r in response.json()]


# Test our function works as expected
get_comments(2792)

["@albertvillanova my tests are failing here:\r\n```\r\ndataset_name = 'gooaq'\r\n\r\n    def test_load_dataset(self, dataset_name):\r\n        configs = self.dataset_tester.load_all_configs(dataset_name, is_local=True)[:1]\r\n>       self.dataset_tester.check_load_dataset(dataset_name, configs, is_local=True, use_local_dummy_data=True)\r\n\r\ntests/test_dataset_common.py:234: \r\n_ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ \r\ntests/test_dataset_common.py:187: in check_load_dataset\r\n    self.parent.assertTrue(len(dataset[split]) > 0)\r\nE   AssertionError: False is not true\r\n```\r\nWhen I try loading dataset on local machine it works fine. Any suggestions on how can I avoid this error?",
 'Thanks for the help, @albertvillanova! All tests are passing now.']

In [30]:
# Depending on your internet connection, this can take a few minutes...
issues_with_comments_dataset = issues_dataset.map(
    lambda x: {"comments": get_comments(x["number"])}
)

Map:   0%|          | 0/8838 [00:00<?, ? examples/s]

In [31]:
from huggingface_hub import notebook_login

notebook_login()

In [32]:
issues_with_comments_dataset.push_to_hub("github-issues")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/tmp8tv5glyk.parquet    :   4%|3         | 1.56MB / 41.5MB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/datasets/pluplu68/github-issues/commit/eb93f761e0e800d413a8620c37ca969e4378374e', commit_message='Upload dataset', commit_description='', oid='eb93f761e0e800d413a8620c37ca969e4378374e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/pluplu68/github-issues', endpoint='https://huggingface.co', repo_type='dataset', repo_id='pluplu68/github-issues'), pr_revision=None, pr_num=None)

In [33]:
remote_dataset = load_dataset("pluplu68/github-issues", split="train")
remote_dataset

README.md:   0%|          | 0.00/7.70k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 41.5MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8838 [00:00<?, ? examples/s]

Dataset({
    features: ['url', 'repository_url', 'labels_url', 'comments_url', 'events_url', 'html_url', 'id', 'node_id', 'number', 'title', 'user', 'labels', 'state', 'locked', 'assignees', 'milestone', 'comments', 'created_at', 'updated_at', 'closed_at', 'assignee', 'author_association', 'issue_field_values', 'type', 'active_lock_reason', 'draft', 'pull_request', 'body', 'closed_by', 'reactions', 'timeline_url', 'performed_via_github_app', 'state_reason', 'sub_issues_summary', 'issue_dependencies_summary', 'pinned_comment', 'is_pull_request'],
    num_rows: 8838
})